## Importações

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Definições das fatias e métricas
deciles = [f"decil_{i}" for i in range(1, 11)]  # decil_1 até decil_10
metrics = ["hrm", "pozzi"]
years = range(2014, 2025)
k_values = [5]

portfolios = {}
market_returns_dict = {}

for year in years:
    for k in k_values:
        path_k1 = f"../../data/02_clean/returns_new_{year-k+1}_{year}.parquet"
        
        if os.path.exists(path_k1):
            full_ret = pd.read_parquet(path_k1)
        else:
            print(f"Arquivo de retornos para o ano {year} não encontrado. Pulando...")
            continue

        # Retorno do mercado (média de todos os ativos do período)
        market_returns_dict[f"{year}_{k}"] = np.log1p(full_ret).mean(axis=1)
        
        # 2. Loop para ler cada decil de cada métrica
        for metric in metrics:
            for decil in deciles:
                # O nome do arquivo que salvamos antes: ex: decil_1_2024_hrm.parquet
                file_path = f"../../data/06_portfolios/{decil}_{year}_{metric}.parquet"
                
                if os.path.exists(file_path):
                    # Lemos o arquivo do decil para pegar a lista de colunas (tickers)
                    portfolio_tickers = pd.read_parquet(file_path).columns.tolist()
                    
                    # Garante que as colunas existam no arquivo de retornos full_ret
                    valid_cols = [c for c in portfolio_tickers if c in full_ret.columns]
                    
                    # Armazena os retornos históricos apenas desses ativos
                    # Chave: decil_1_2024_5_hrm
                    portfolios[f"{decil}_{year}_{k}_{metric}"] = full_ret[valid_cols]
                else:
                    # Opcional: print para debug se faltar algum decil
                    # print(f"Aviso: {file_path} não encontrado.")
                    pass

/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/

In [2]:
weekly_returns = {}
weekly2_returns = {}
monthly_returns = {}
daily_returns = {}

weighting = "equal"

# Carrega metadados uma única vez
# df_mcap = pd.read_csv("../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv")

for name, portfolio in portfolios.items():
    # Ajuste no split: decil_1_2014_5_hrm -> ['decil', '1', '2014', '5', 'hrm']
    parts = name.split('_')
    
    # Reconstruindo a lógica de decil e métrica
    decil_label = f"{parts[0]}_{parts[1]}" # 'decil_1'
    year = int(parts[2])
    k = int(parts[3])
    metric = parts[4] # 'hrm' ou 'pozzi'
    
    # Log returns do portfólio
    log_returns = np.log1p(portfolio)
    
    # Market returns correspondente (precisa da chave ano_k)
    R_m = market_returns_dict[f"{year}_{k}"]
    
    # --- Lógica de Pesos ---
    portfolio_log_return = log_returns.mean(axis=1)
    
    # --- Armazenamento dos Retornos em diferentes frequências ---
    # Chave original preservada (decil_X_ano_k_metrica)
    daily_returns[name] = portfolio_log_return
    
    # Semanais (Sexta-feira)
    weekly_returns[name] = portfolio_log_return.resample("W-FRI").sum()
    
    # Quinzenais
    weekly2_returns[name] = portfolio_log_return.resample("2W-FRI").sum()
    
    # Mensais (Corrigido para usar resample mensal)
    monthly_returns[name] = portfolio_log_return.resample("ME").sum()

print("Processamento concluído para todos os decis e métricas.")

/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/

Processamento concluído para todos os decis e métricas.


/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


## Lead-lag

In [3]:
import matplotlib.pyplot as  plt
import seaborn as sns

def autocorrelation_matrix(X, lag):
    X_t = X.iloc[lag:]
    X_tk = X.shift(lag).iloc[lag:]

    mu_t = X_t.mean().values
    mu_tk = X_tk.mean().values

    Xc_t = X_t.values - mu_t
    Xc_tk = X_tk.values - mu_tk

    Sigma_k = (Xc_tk.T @ Xc_t) / len(Xc_t)

    std_t = X_t.std(ddof=0).values
    std_tk = X_tk.std(ddof=0).values

    # Avoid division by zero
    std_t[std_t == 0] = 1e-8
    std_tk[std_tk == 0] = 1e-8

    return Sigma_k / np.outer(std_tk, std_t)

def plot_antisymmetric_autocorr(
    returns_list,
    column_names,
    labels,
    lags=(1, 2, 3, 4),
    figsize=(12, 8),
    cmap="Blues",
    title_prefix="Y",
    annot=False,
    diff=True,
    fig_title=None
):
    X = pd.concat(returns_list, axis=1).dropna()
    X.columns = column_names

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.flatten()

    if len(lags) == 1:
        axes = [axes]

    for ax, lag in zip(axes, lags):
        T = autocorrelation_matrix(X, lag)

        if diff:
            A = pd.DataFrame(
                T - T.T,
                index=labels,
                columns=labels
            )
            title = f"{title_prefix}({lag}) - {title_prefix}'({lag})"
        else:
            A = pd.DataFrame(
                T,
                index=labels,
                columns=labels
            )
            title = f"{title_prefix}({lag})"

        sns.heatmap(A, ax=ax, cmap=cmap, center=0, annot=annot)
        ax.set_title(title)

    if fig_title:
        fig.suptitle(fig_title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [4]:
### Central Peripheral

In [5]:
import os
import pandas as pd
import numpy as np

# Definições
metrics = ["hrm", "pozzi"]
cols_names = ["cp1d", "cp1w", "cp2w", "cp1m"]
max_lag = 5

# Pares de decis: (1,10), (2,9), (3,8), (4,7), (5,6)
decile_pairs = [(i, 11 - i) for i in range(1, 6)]

for year in years:
    for k in k_values:
        for metric in metrics:
            for p_idx, c_idx in decile_pairs:
                
                # Gerando os labels dinamicamente
                # Ex: decil_1 vs decil_10
                p_label = f"decil_{p_idx}_{year}_{k}_{metric}"
                c_label = f"decil_{c_idx}_{year}_{k}_{metric}"

                try:
                    freq_data = {
                        "1d": (daily_returns[p_label], daily_returns[c_label]),
                        "1w": (weekly_returns[p_label], weekly_returns[c_label]),
                        "2w": (weekly2_returns[p_label], weekly2_returns[c_label]),
                        "1m": (monthly_returns[p_label], monthly_returns[c_label])
                    }
                except KeyError:
                    # Silencioso para não poluir o terminal, ou use um print se preferir debug
                    continue

                matrix_results = []

                for freq in ["1d", "1w", "2w", "1m"]:
                    R_low, R_high = freq_data[freq]
                    
                    X = pd.concat([R_low, R_high], axis=1).dropna()
                    X.columns = ["Low_Decile", "High_Decile"]
                    
                    valid_lags = [l for l in range(1, max_lag + 1) if len(X) >= l + 4]
                    
                    if not valid_lags:
                        matrix_results.append(np.nan)
                        continue
                    
                    total_lead_lag = 0
                    for l in valid_lags:
                        acm = autocorrelation_matrix(X, lag=l)
                        lag_matrix = acm - acm.T
                        
                        if metric == "hrm":
                            total_lead_lag += lag_matrix[0, 1]
                        else: 
                            total_lead_lag += lag_matrix[0, 1]
                    
                    matrix_results.append(total_lead_lag)

                # Criando o DataFrame
                leadlag_df = pd.DataFrame(
                    [matrix_results],
                    columns=cols_names,
                    index=[year]
                )

                # Nome do arquivo agora identifica os decis: ex: leadlag_hrm_2024_d1_d10.csv
                os.makedirs("../../data/08_lead_lag", exist_ok=True)
                filename = f"leadlag_{metric}_{year}_{k}_d{p_idx}_d{c_idx}.csv"
                leadlag_df.to_csv(f"../../data/08_lead_lag/{filename}")

### Lead-lag considerando Market Cap

In [6]:
import pandas as pd
import numpy as np

years = range(2014, 2025)

# Load metadata and returns once outside the loop for efficiency
df_mcap_meta = pd.read_csv(
    "../../data/02_clean/metadata - metadata_att (1).csv"
)
df_ret_full = pd.read_parquet("../../data/02_clean/returns_30_years.parquet")
df_ret_full = df_ret_full[df_ret_full.index >= pd.Timestamp(2015, 1, 1)]

for year in years:
    try:
        # 1. Load returns for the calculation window
        returns_prev = pd.read_parquet(f"../../data/02_clean/returns_new_{year-4}_{year}.parquet")
        
        # 2. Market cap cleaning and decile logic
        df_mcap_year = df_mcap_meta[(df_mcap_meta["Ticker"].isin(returns_prev.columns)) &
                                    (df_mcap_meta["Ticker"].isin(df_ret_full.columns))].copy()
        mcap_col = f"mcap_{year}"

        # Clean numeric data (handle dots and commas)
        df_mcap_year[mcap_col] = (
            df_mcap_year[mcap_col]
            .replace("#ERROR!", np.nan)
            .astype(str)
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        df_mcap_year[mcap_col] = pd.to_numeric(df_mcap_year[mcap_col], errors="coerce")
        df_mcap_year = df_mcap_year.dropna(subset=[mcap_col])

        # Define Portfolios based on deciles (Decil 1 and 10)
        p90 = df_mcap_year[mcap_col].quantile(0.9) # Threshold for Decile 10 (Largest)
        p10 = df_mcap_year[mcap_col].quantile(0.1) # Threshold for Decile 1 (Smallest)

        large_tickers = df_mcap_year[df_mcap_year[mcap_col] >= p90]["Ticker"].tolist()
        small_tickers = df_mcap_year[df_mcap_year[mcap_col] <= p10]["Ticker"].tolist()

        # 3. Calculate Portfolio Returns (Log returns)
        # R1/R2: Daily
        log_rets = np.log1p(returns_prev)
        R1 = log_rets[large_tickers].mean(axis=1) # Large (Decile 10)
        R2 = log_rets[small_tickers].mean(axis=1) # Small (Decile 1)

        # R3/R4: Weekly (Friday)
        R3 = log_rets[large_tickers].resample("W-FRI").sum().mean(axis=1)
        R4 = log_rets[small_tickers].resample("W-FRI").sum().mean(axis=1)

        # R5/R6: 2-Week
        R5 = log_rets[large_tickers].resample("2W-FRI").sum().mean(axis=1)
        R6 = log_rets[small_tickers].resample("2W-FRI").sum().mean(axis=1)

        # R7/R8: Monthly
        R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
        R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)

        # 4. Lead-Lag Matrix Calculation
        matrix = []
        lag = 1
        
        # Pairs to iterate through (Large vs Small at different frequencies)
        pairs = [(R1, R2), (R3, R4), (R5, R6), (R7, R8)]
        
        for large_ret, small_ret in pairs:
            X = pd.concat([large_ret, small_ret], axis=1).dropna()
            X.columns = ["Large", "Small"]
            
            acm = autocorrelation_matrix(X, lag)
            
            # Cross-autocorrelation asymmetry: (Large leads Small) - (Small leads Large)
            lag_matrix = acm - acm.T
            matrix.append(lag_matrix[0, 1])

        # 5. Save Results
        cols = ["ls1d", "ls1w", "ls2w", "ls1m"]
        leadlag_df = pd.DataFrame([matrix], columns=cols, index=[year])
        leadlag_df.to_csv(f"../../data/08_lead_lag/marketcap_leadlag_df_{year}.csv")
        
    except Exception as e:
        print(f"Error processing year {year}: {e}")
        continue

/tmp/ipykernel_12074/2856094786.py:56: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
/tmp/ipykernel_12074/2856094786.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)
/tmp/ipykernel_12074/2856094786.py:56: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
/tmp/ipykernel_12074/2856094786.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)
/tmp/ipykernel_12074/2856094786.py:56: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R7 = log_rets[large_tickers].resample("M").sum().mea